# Finance & Investment Risk Analysis
### Sector Trends and Loan Default Risk: A Statistical Study


This notebook consolidates Phases 0–10 of the project: data cleaning, exploratory analysis,
hypothesis testing, and regression modelling across S&P 500 sector data and consumer loan data.

In [3]:
%matplotlib inline
import sys, os
sys.path.append('..')
os.chdir('..')  # so relative paths like "data/raw/..." resolve from project root, matching how scripts run via `python src/xxx.py`
print("Working directory:", os.getcwd())

Working directory: d:\Data Analytics\finance-investment-risk-analysis


## 2. Business Problem

Smart Analytics Inc. is simulating a consulting engagement covering two linked finance risk questions:
- **Market-side risk**: how do S&P 500 sector returns vary, and how correlated are sectors (diversification)?
- **Credit-side risk**: does borrower income (and related variables) predict loan default?

## 3. Objectives

- Clean and merge the S&P 500 price/sector data and the loan datasets
- Characterise sector-wise return behaviour and detect outliers (IQR)
- Quantify sector co-movement via correlation/pair plots
- Demonstrate the Central Limit Theorem empirically
- Formally test income differences between defaulters and non-defaulters
- Fit linear, logistic, spline, and Poisson regression models where each is genuinely appropriate
- Simulate an observational policy backtest (DTI threshold)
- Translate every result into a business recommendation

## 4. Dataset Description

| Dataset | Rows | Key columns |
|---|---|---|
| `sp500_data.csv` | wide, 1993–2015 | date, ticker columns (daily $ price change) |
| `sp500_sectors.csv` | 517 | symbol, sector, sector_label, sub_sector |
| `loan_data.csv` | 45,342 | annual_inc, loan_amnt, dti, outcome, borrower_score, ... |
| `loan3000.csv` | 3,000 | outcome, dti, borrower_score, payment_inc_ratio (no income) |

**Note:** S&P 500 values are **daily dollar price changes** (Close_t − Close_t-1), not percentage returns.
The loan `outcome` variable is an exact 50/50 balanced sample, not the true real-world default rate.

## 5–6. Data Loading & Cleaning

### S&P 500 pipeline (Phase 1)
Run once — produces `data/processed/sp500_price_changes_long.pkl` and
`data/processed/sp500_sectors_clean.pkl`, reused by every S&P 500 phase below.

In [4]:
from src.sp500_pipeline import run_pipeline

pipeline_result = run_pipeline(save=True)
for k, v in pipeline_result["report"].items():
    print(f"{k}: {v}")

sector_map_shape: (517, 5)
price_wide_raw_shape: (5647, 517)
total_pre_listing_cells_removed: 413549
tickers_with_no_data_at_all: 0
long_shape_before_sector_merge: (2505950, 3)
long_shape_after_sector_merge: (2505950, 7)
unmatched_symbols: 0


### Loan data
Loaded directly from `data/raw/` in each phase below (no separate cleaning pipeline needed —
both loan files were confirmed clean on inspection: 0 missing values, 0 duplicates).

## 7–11. S&P 500 EDA — Sector Composition, Return & Risk, Percentiles/IQR, Correlation & Pair Plots (Phase 2)

In [5]:
from src.sp500_eda import (
    load_processed_data, summarize_dataset, analyze_sector_distribution,
    analyze_price_changes, detect_iqr_outliers, analyze_sector_statistics,
    create_correlation_analysis, create_pairplot
)

prices, sectors = load_processed_data()
summarize_dataset(prices, sectors)

DATASET OVERVIEW
Observations (ticker-days) : 2,505,950
Unique tickers              : 517
  - Regular stocks          : 500
  - ETFs                    : 17
Sector labels (incl. 'EFTs'): 11
Date range                  : 1993-01-29 to 2015-07-01

Observations per sector_label:
sector_label
Financials                442333
Consumer Discretionary    386430
Industrials               341684
Technology                296061
Health Care               271494
Energy                    192905
Consumer Staples          185914
Utilities                 157206
Materials                 141917
EFTs                       63078
Telecom                    26928
Name: count, dtype: int64


{'n_observations': 2505950,
 'n_unique_tickers': 517,
 'n_stock_tickers': 500,
 'n_etf_tickers': 17,
 'n_sectors_incl_etf_bucket': 11,
 'date_min': Timestamp('1993-01-29 00:00:00'),
 'date_max': Timestamp('2015-07-01 00:00:00')}

In [6]:
analyze_sector_distribution(sectors)


Sector composition:
          sector_label  n_tickers  pct
            Financials         88 17.6
Consumer Discretionary         84 16.8
           Industrials         68 13.6
            Technology         66 13.2
           Health Care         56 11.2
                Energy         40  8.0
      Consumer Staples         36  7.2
             Utilities         29  5.8
             Materials         28  5.6
               Telecom          5  1.0
                  EFTs          0  0.0


,sector_label,n_tickers,pct
0,Financials,88,17.6
1,Consumer Discretionary,84,16.8
2,Industrials,68,13.6
3,Technology,66,13.2
4,Health Care,56,11.2
5,Energy,40,8.0
6,Consumer Staples,36,7.2
7,Utilities,29,5.8
8,Materials,28,5.6
9,Telecom,5,1.0


In [7]:
analyze_price_changes(prices)


Daily price-change distribution (stocks only, in $):
count    2.442872e+06
mean     1.508698e-03
std      1.350434e+00
min     -2.137500e+02
1%      -2.349975e+00
5%      -9.200000e-01
25%     -1.854327e-01
50%      0.000000e+00
75%      1.999740e-01
95%      9.299980e-01
99%      2.292465e+00
max      1.865625e+02
Name: price_change, dtype: float64


count    2.442872e+06
mean     1.508698e-03
std      1.350434e+00
min     -2.137500e+02
1%      -2.349975e+00
5%      -9.200000e-01
25%     -1.854327e-01
50%      0.000000e+00
75%      1.999740e-01
95%      9.299980e-01
99%      2.292465e+00
max      1.865625e+02
Name: price_change, dtype: float64

In [8]:
detect_iqr_outliers(prices)


IQR outlier analysis (daily $ price change, stocks only):
  Q1: -0.1854
  Q3: 0.2000
  IQR: 0.3854
  lower_fence: -0.7635
  upper_fence: 0.7781
  n_outliers: 320075
  pct_outliers: 13.1024
  n_total: 2442872


{'Q1': -0.18543265014886856,
 'Q3': 0.19997401535511017,
 'IQR': 0.38540666550397873,
 'lower_fence': -0.7635426484048367,
 'upper_fence': 0.7780840136110783,
 'n_outliers': 320075,
 'pct_outliers': 13.102405692971224,
 'n_total': 2442872}

In [9]:
analyze_sector_statistics(prices)


Sector-level daily $ price-change statistics:
                          mean  median     std      q1      q3   n_obs     iqr
sector_label                                                                  
Telecom                -0.1434     0.0  7.0868 -0.1378  0.1200   26928  0.2578
Financials             -0.0002     0.0  1.7074 -0.1869  0.2050  442333  0.3918
Consumer Discretionary  0.0019     0.0  1.4127 -0.1968  0.2041  386430  0.4009
Technology             -0.0059     0.0  1.1651 -0.2296  0.2368  296061  0.4665
Health Care             0.0138     0.0  1.0072 -0.2085  0.2312  271494  0.4397
Energy                 -0.0083     0.0  0.7574 -0.2147  0.2143  192905  0.4290
Materials               0.0057     0.0  0.6679 -0.2120  0.2260  141917  0.4380
Industrials             0.0085     0.0  0.6360 -0.1852  0.2065  341684  0.3918
Consumer Staples        0.0115     0.0  0.4294 -0.1338  0.1546  185914  0.2884
Utilities               0.0040     0.0  0.3341 -0.1078  0.1226  157206  0.2304


,mean,median,std,q1,q3,n_obs,iqr
sector_label,,,,,,,
Telecom,-0.1434,0.0,7.0868,-0.1378,0.1200,26928,0.2578
Financials,-0.0002,0.0,1.7074,-0.1869,0.2050,442333,0.3918
Consumer Discretionary,0.0019,0.0,1.4127,-0.1968,0.2041,386430,0.4009
Technology,-0.0059,0.0,1.1651,-0.2296,0.2368,296061,0.4665
Health Care,0.0138,0.0,1.0072,-0.2085,0.2312,271494,0.4397
Energy,-0.0083,0.0,0.7574,-0.2147,0.2143,192905,0.4290
Materials,0.0057,0.0,0.6679,-0.2120,0.2260,141917,0.4380
Industrials,0.0085,0.0,0.6360,-0.1852,0.2065,341684,0.3918
Consumer Staples,0.0115,0.0,0.4294,-0.1338,0.1546,185914,0.2884


In [10]:
create_correlation_analysis(prices)


Representative subset for correlation (20 tickers, 2 per sector, chosen by longest available history):
['BEN', 'CA', 'CL', 'CLX', 'CNP', 'CTL', 'ESV', 'GAS', 'GLW', 'HAS', 'HRB', 'MAT', 'MSFT', 'NEM', 'NUE', 'PBI', 'T', 'UHS', 'VLO', 'XRAY']


symbol,BEN,CA,CL,CLX,CNP,CTL,ESV,GAS,GLW,HAS,HRB,MAT,MSFT,NEM,NUE,PBI,T,UHS,VLO,XRAY
symbol,,,,,,,,,,,,,,,,,,,,
BEN,1.000000,0.233104,0.355196,0.318035,0.259225,0.322854,0.316494,0.345069,0.172443,0.339606,0.378067,0.309964,0.355325,0.161313,0.454422,0.309357,0.329935,0.287435,0.331326,0.408742
CA,0.233104,1.000000,0.136719,0.121693,0.125562,0.171308,0.143308,0.145544,0.195076,0.187738,0.178188,0.213620,0.332382,0.069533,0.156397,0.240478,0.190400,0.088022,0.109273,0.178792
CL,0.355196,0.136719,1.000000,0.516537,0.230587,0.247651,0.163371,0.287126,0.034896,0.243648,0.260270,0.219066,0.265238,0.084297,0.206419,0.194655,0.288375,0.209987,0.124071,0.269589
CLX,0.318035,0.121693,0.516537,1.000000,0.219284,0.217885,0.143309,0.287433,0.073938,0.230474,0.247273,0.211739,0.239066,0.047583,0.204055,0.200683,0.271727,0.189433,0.146473,0.244228
CNP,0.259225,0.125562,0.230587,0.219284,1.000000,0.168216,0.197632,0.419778,-0.003042,0.188989,0.189978,0.154736,0.185554,0.084652,0.199101,0.198814,0.220975,0.145766,0.186611,0.210907
CTL,0.322854,0.171308,0.247651,0.217885,0.168216,1.000000,0.184497,0.280194,0.120732,0.233622,0.238799,0.223162,0.261725,0.091354,0.227713,0.240325,0.405853,0.144897,0.153510,0.236711
ESV,0.316494,0.143308,0.163371,0.143309,0.197632,0.184497,1.000000,0.269543,0.109444,0.184335,0.196396,0.161735,0.210928,0.244555,0.406309,0.178974,0.177143,0.151349,0.380637,0.230951
GAS,0.345069,0.145544,0.287126,0.287433,0.419778,0.280194,0.269543,1.000000,0.106463,0.262896,0.272897,0.218958,0.253179,0.135857,0.306229,0.235575,0.247027,0.235395,0.271902,0.307287
GLW,0.172443,0.195076,0.034896,0.073938,-0.003042,0.120732,0.109444,0.106463,1.000000,0.107322,0.110592,0.085283,0.228049,0.028563,0.133797,0.164538,0.103897,0.069781,0.093652,0.098059


In [11]:
create_pairplot(prices)


Pair plot tickers (largest-coverage stock per target sector): ['AAPL', 'AFL', 'APA', 'ABT', 'ADM']


**Key findings:** Financials (17.6%) and Consumer Discretionary (16.8%) are the largest sectors.
13.1% of daily price changes are IQR-flagged outliers (expected — fat-tailed daily returns, not data errors).
Cross-sector correlations stayed low (max 0.52, and that pair was *within* Consumer Staples), supporting
genuine diversification benefit across sectors.

## 12. Central Limit Theorem Demonstration (Phase 4)

In [12]:
from src.clt_demo import load_data, population_stats, run_clt_simulation, plot_clt_grid, plot_se_convergence

income = load_data()
pop_stats = population_stats(income)
summary, sampling_results = run_clt_simulation(income)
plot_clt_grid(income, sampling_results, pop_stats["mu"], pop_stats["sigma"])
plot_se_convergence(summary)

POPULATION (raw income) STATISTICS
Population mean (mu)     : $68,211.70
Population std (sigma)   : $56,239.02
Population skewness      : 46.825  (0 = symmetric; positive = right-skewed, matches Phase 3 finding)

CLT SIMULATION SUMMARY
(population mu=$68,211.70, sigma=$56,239.02, 1000 repetitions per n)

  n  reps  mean_of_sample_means  theoretical_se_sigma_over_sqrt_n  empirical_se_of_sample_means  skewness_of_sampling_dist  pct_reps_containing_gt300k_earner
 10  1000             67694.963                         17784.341                     13304.131                      1.343                                2.4
 30  1000             67819.400                         10267.794                     10809.154                     10.338                                8.4
 50  1000             67840.315                          7953.399                      7486.762                      6.282                               13.7
100  1000             68174.366                          5623.

**Key finding:** the sampling distribution's bulk visibly tightens and centers on μ as n grows,
but convergence is slower than the textbook "n=30" rule — even at n=100, ~29% of simulated samples
included a >$300k earner, reflecting the population's extreme skewness (skew ≈ 47).

## 13–14. Loan EDA — Income Distribution & Income vs. Default (Phase 3)

In [13]:
from src.loan_eda import (
    load_loan_data, summarize_loan_dataset, analyze_income_distribution,
    analyze_default_distribution, analyze_income_vs_default, detect_income_outliers,
    analyze_loan_correlations, create_loan_pairplot
)

loan_data, loan3000 = load_loan_data()
summarize_loan_dataset(loan_data, loan3000)

LOAN DATASET OVERVIEW
loan_data.csv : 45,342 rows x 20 cols
loan3000.csv  : 3,000 rows x 5 cols (no income column -> reference only in this phase)

Missing values in loan_data : 0
Duplicate rows in loan_data  : 0

Outcome balance in loan_data:
outcome
default     22671
paid off    22671
Name: count, dtype: int64
NOTE: exact 50/50 split -> this is a balanced (likely undersampled)
dataset, not the true population default rate. Any 'X% default rate'
statement about the real world should NOT be based on this split.


{'loan_data_rows': 45342,
 'loan3000_rows': 3000,
 'missing_values': 0,
 'duplicate_rows': 0,
 'n_default': 22671,
 'n_paid_off': 22671}

In [14]:
analyze_income_distribution(loan_data)


Annual income distribution ($):
count    4.534200e+04
mean     6.821170e+04
std      5.623902e+04
min      2.000000e+03
1%       1.560000e+04
5%       2.500000e+04
25%      4.200000e+04
50%      6.000000e+04
75%      8.100000e+04
95%      1.400000e+05
99%      2.250000e+05
max      7.141778e+06
Name: annual_inc, dtype: float64


count    4.534200e+04
mean     6.821170e+04
std      5.623902e+04
min      2.000000e+03
1%       1.560000e+04
5%       2.500000e+04
25%      4.200000e+04
50%      6.000000e+04
75%      8.100000e+04
95%      1.400000e+05
99%      2.250000e+05
max      7.141778e+06
Name: annual_inc, dtype: float64

In [15]:
analyze_default_distribution(loan_data)


Default vs non-default counts:
  default: 22,671 (50.0%)
  paid off: 22,671 (50.0%)


outcome
default     22671
paid off    22671
Name: count, dtype: int64

In [16]:
analyze_income_vs_default(loan_data)


Income by outcome:
            count          mean           std     min      25%      50%  \
outcome                                                                   
default   22671.0  63586.725641  41108.615664  2000.0  40000.0  55000.0   
paid off  22671.0  72836.666490  67772.372355  3500.0  45000.0  62000.0   

              75%        max  
outcome                       
default   75000.0   932000.0  
paid off  87113.0  7141778.0  


,count,mean,std,min,25%,50%,75%,max
outcome,,,,,,,,
default,22671.0,63586.725641,41108.615664,2000.0,40000.0,55000.0,75000.0,932000.0
paid off,22671.0,72836.666490,67772.372355,3500.0,45000.0,62000.0,87113.0,7141778.0


In [17]:
detect_income_outliers(loan_data)


IQR outlier analysis (annual income):
  Q1: 42,000.00
  Q3: 81,000.00
  IQR: 39,000.00
  lower_fence: 0
  upper_fence: 139,500.00
  n_outliers: 2300
  pct_outliers: 5.07
  n_total: 45342

DECISION: high-income outliers are RETAINED, not removed or capped.
They are almost certainly genuine high earners, and income is a
candidate PREDICTOR for later regression/hypothesis testing —
silently deleting real high-income borrowers would bias that
analysis rather than clean it. Any single extreme point (e.g. the
$7.14M row) should be sanity-checked individually before it is
allowed to drive a regression fit.


{'Q1': 42000.0,
 'Q3': 81000.0,
 'IQR': 39000.0,
 'lower_fence': 0,
 'upper_fence': 139500.0,
 'n_outliers': 2300,
 'pct_outliers': 5.072559657712496,
 'n_total': 45342}

In [18]:
analyze_loan_correlations(loan_data)


Correlation matrix (key numeric variables):
                   annual_inc  loan_amnt   dti  payment_inc_ratio  revol_bal  \
annual_inc               1.00       0.31 -0.16              -0.26       0.26   
loan_amnt                0.31       1.00  0.08               0.52       0.29   
dti                     -0.16       0.08  1.00               0.23       0.16   
payment_inc_ratio       -0.26       0.52  0.23               1.00       0.00   
revol_bal                0.26       0.29  0.16               0.00       1.00   
revol_util               0.02       0.10  0.25               0.11       0.21   
grade                   -0.02      -0.25 -0.12              -0.19      -0.05   
borrower_score           0.05      -0.02 -0.23              -0.10      -0.05   

                   revol_util  grade  borrower_score  
annual_inc               0.02  -0.02            0.05  
loan_amnt                0.10  -0.25           -0.02  
dti                      0.25  -0.12           -0.23  
payment_inc_ra

,annual_inc,loan_amnt,dti,payment_inc_ratio,revol_bal,revol_util,grade,borrower_score
annual_inc,1.000000,0.310525,-0.156245,-0.262502,0.259401,0.017311,-0.020360,0.052661
loan_amnt,0.310525,1.000000,0.075155,0.522273,0.285700,0.099535,-0.249466,-0.023900
dti,-0.156245,0.075155,1.000000,0.231693,0.159050,0.248429,-0.121441,-0.227145
payment_inc_ratio,-0.262502,0.522273,0.231693,1.000000,0.002187,0.114169,-0.186675,-0.103581
revol_bal,0.259401,0.285700,0.159050,0.002187,1.000000,0.207604,-0.051111,-0.054003
revol_util,0.017311,0.099535,0.248429,0.114169,0.207604,1.000000,-0.327684,-0.458232
grade,-0.020360,-0.249466,-0.121441,-0.186675,-0.051111,-0.327684,1.000000,0.207415
borrower_score,0.052661,-0.023900,-0.227145,-0.103581,-0.054003,-0.458232,0.207415,1.000000


In [19]:
create_loan_pairplot(loan_data)


Pair plot: 1983-row random sample, variables = ['annual_inc', 'loan_amnt', 'dti', 'borrower_score'], coloured by outcome


**Key findings:** Non-defaulters earn more on average ($72,837 vs $63,587). 5.07% of income values
are IQR outliers, all high-income and retained (income is a planned regression predictor — deleting
real high earners would bias, not clean, the analysis).

## 15. Hypothesis Testing (Phase 5)

Three tests: two-sample and one-tailed t-tests on income by default status, and a sector-returns t-test
(Consumer Staples vs. Technology, chosen *a priori* from Phase 2's finding — not cherry-picked).

In [20]:
from src.hypothesis_tests import (
    load_loan_data as load_loan_data_h5, load_sp500_data,
    two_sample_income_ttest, one_tailed_income_ttest, sector_returns_ttest
)

loan_data_h5 = load_loan_data_h5()
sp500_prices_h5 = load_sp500_data()

res1 = two_sample_income_ttest(loan_data_h5)
res2 = one_tailed_income_ttest(loan_data_h5, equal_var=res1["equal_var_assumed"])
res3 = sector_returns_ttest(sp500_prices_h5)

TEST 1: TWO-SAMPLE T-TEST -- INCOME BY DEFAULT STATUS
H0: mu_default = mu_non_default
H1: mu_default != mu_non_default (two-tailed)
alpha = 0.05

Levene's test for equal variances: stat=83.852, p=5.548e-20
  -> Variances differ significantly, using Welch's t-test (unequal variance)

Defaulters   : mean=$63,586.73, n=22,671
Non-defaults : mean=$72,836.67, n=22,671
t-statistic  : -17.5708
p-value      : 7.814e-69
Decision     : Reject H0 at alpha=0.05

TEST 2: ONE-TAILED T-TEST -- DEFAULTERS HAVE LOWER INCOME
H0: mu_default >= mu_non_default
H1: mu_default <  mu_non_default
alpha = 0.05

Sample difference is in the H1 direction: True
t-statistic       : -17.5708
one-tailed p-value: 3.907e-69
Decision          : Reject H0 at alpha=0.05

TEST 3: TWO-SAMPLE T-TEST -- Consumer Staples vs Technology DAILY PRICE CHANGE
H0: mu_Consumer Staples = mu_Technology
H1: mu_Consumer Staples != mu_Technology (two-tailed)
alpha = 0.05

Levene's test for equal variances: stat=8892.525, p=0
  -> using Welc

**Key findings:** Both income tests reject H0 (Welch's t=-17.57, p=7.8e-69) — defaulters earn
significantly less. Sector returns test also rejects H0 (t=7.34, p=2.1e-13), but the effect size
(~1.7 cents/day) is tiny — a case of large sample size inflating statistical significance without
implying large practical significance.

## 16. Regression Analysis (Phase 6)

`regression.py` is a flat top-level script (not function-wrapped) — run via `%run` rather than imported,
per the "deliver exact final code, no unsolicited rewrites" rule.

In [22]:
%run src/regression.py

                            OLS Regression Results                            
Dep. Variable:              loan_amnt   R-squared:                       0.096
Model:                            OLS   Adj. R-squared:                  0.096
Method:                 Least Squares   F-statistic:                     4838.
Date:                Sat, 15 Aug 2026   Prob (F-statistic):               0.00
Time:                        01:23:46   Log-Likelihood:            -4.7003e+05
No. Observations:               45342   AIC:                         9.401e+05
Df Residuals:                   45340   BIC:                         9.401e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       1.011e+04     56.762    178.086      0.0

<Figure size 704x528 with 0 Axes>

**Key findings:** Linear regression (loan_amnt ~ annual_inc): R²=0.096, coef=0.0447 (p<0.001).
Logistic regression (default ~ income + loan_amnt + dti + revol_util + borrower_score): ROC-AUC=0.686,
McFadden pseudo-R²=0.076. `borrower_score` dominates; `revol_util` not significant once other
variables are controlled for.

## 17. Spline Regression (Phase 7)

Only applied because decile-binned means confirmed genuine nonlinearity in Phase 6's residuals
(diminishing-returns shape between income and loan amount) — not used to satisfy a checklist.

In [23]:
%run src/spline_regression.py

=== Decile-binned means (income vs loan amount) ===
   income_decile    mean_income  mean_loan_amt     n
0              0   24191.488644    6033.566703  4535
1              1   35248.783778    8802.970024  4537
2              2   42361.962157   10571.724313  4915
3              3   48769.180946   11674.262059  4167
4              4   56440.917780   12838.512026  6361
5              5   63650.916636   13869.581326  2699
6              6   71146.234029   14808.824478  4649
7              7   83195.037773   16344.209742  5030
8              8  100270.602197   17777.114205  3914
9              9  163018.489085   19585.584344  4535

=== Model comparison: loan_amnt ~ f(annual_inc) ===
Model               R2        RMSE        AIC         
Linear              0.0964    7688.60     940073.5    
Log-income          0.2334    7081.71     932617.2    
Natural spline(4df) 0.2160    7161.65     933641.2    


**Key finding:** log-income transform (R²=0.233) outperformed both plain linear (R²=0.096)
and a natural cubic spline (R²=0.216) on every metric — a genuine Occam's razor result.

## 18. Poisson Regression (Phase 8)

`open_acc` (number of open credit accounts) was the only genuine count variable found across both
loan datasets — used here as a defensible Poisson target.

In [24]:
%run src/poisson_regression.py

=== Overdispersion check ===
Mean(open_acc): 10.4131
Variance(open_acc): 22.0582
Variance/Mean ratio: 2.1183

=== Poisson Regression: open_acc ~ annual_inc + dti + loan_amnt + borrower_score ===
                 Generalized Linear Model Regression Results                  
Dep. Variable:               open_acc   No. Observations:                45342
Model:                            GLM   Df Residuals:                    45337
Model Family:                 Poisson   Df Model:                            4
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.3082e+05
Date:                Sat, 15 Aug 2026   Deviance:                       75846.
Time:                        01:24:09   Pearson chi2:                 8.03e+04
No. Iterations:                    11   Pseudo R-squ. (CS):             0.2962
Covariance Type:            nonrobust                                         
               

**Key finding:** All four predictors significant; `borrower_score` has by far the largest effect
(IRR=1.504). Overdispersion confirmed (variance/mean=2.12, dispersion stat=1.77) and disclosed as an
honest limitation rather than silently corrected, since the assignment scope specifies Poisson.

## 18b. A/B-Style Policy Simulation (Phase 9)

**Framing note:** this is an observational policy backtest, not a true randomized A/B test — no random
assignment occurred in how these historical loans were originally issued.

In [25]:
%run src/ab_policy_simulation.py

DTI median (proposed policy cutoff): 16.02

=== Observational comparison: default rate by DTI group ===
Low DTI (<= median):  n=22687, defaults=10074, default rate=0.4440
High DTI (> median):  n=22655, defaults=12597, default rate=0.5560
Absolute difference in default rate: 0.1120

=== Two-proportion z-test ===
H0: default rate (high DTI) = default rate (low DTI)
H1: default rate (high DTI) > default rate (low DTI)  [one-tailed]
z-statistic: 23.8475
p-value: 5.3776e-126
Decision at alpha=0.05: Reject H0

=== Illustrative policy impact (backtest only, not causal) ===
If loans with DTI > median had all been declined historically:
  Defaults avoided (upper bound, assumes no substitution effect): 12597
  Loans foregone (also would have excluded good borrowers): 10058


**Key finding:** default rate is 44.4% below median DTI vs. 55.6% above (z=23.85, p<0.001).
A stricter DTI cutoff would avoid up to 12,597 defaults but also forgo 10,058 loans that were repaid —
a real precision/recall trade-off requiring a cost-benefit framework, not a statistical answer alone.

## 19. Key Findings

- Cross-sector diversification benefit is modest but real (max pairwise correlation 0.52, and that was *within*-sector)
- Income, DTI, and borrower_score are all statistically robust (p<0.001) predictors of default risk
- Loan-amount-to-income scaling is genuinely nonlinear (log-shaped, not linear)
- Model discriminatory power is moderate (AUC=0.686) — real signal, not a strong standalone classifier
- DTI policy backtest shows a real trade-off, not a free win, between defaults avoided and good loans foregone

## 20. Financial Risk Interpretation

**Market side:** favour cross-sector diversification over intra-sector diversification; sector volatility
comparisons here are dollar-scale, not %-scale, so treat sector risk rankings cautiously without further
normalization.

**Credit side:** income, DTI, and borrower_score are defensible screening inputs but not sufficient alone
(AUC=0.686, ~36.5% misclassification at threshold 0.5). `borrower_score` is the most decisive single variable.

## 21. Recommendations

1. Use borrower_score, income, and DTI together as a multi-factor screening framework — not any single variable alone
2. Favour cross-sector (not intra-sector) diversification for risk-averse portfolios
3. If setting a DTI-based approval cutoff, explicitly weigh the cost of forgone good loans against defaults avoided
4. Normalize sector volatility to % returns before using it for cross-ticker risk comparisons
5. Recalibrate any deployed default model against the true (non-50/50) real-world base rate before use

## 22. Limitations

- All relationships are observational/associational, not causal
- Balanced 50/50 default sample is not the true population default rate
- S&P 500 volatility comparisons are on a dollar, not percentage, scale
- Poisson regression shows overdispersion (dispersion stat=1.77); Negative Binomial would be more correct for production use
- Neither loan dataset includes credit-history variables, which likely limits achievable model performance

## 23. Final Conclusion

This project applied a complete classical statistical pipeline — cleaning, EDA, IQR outlier detection,
correlation/pair-plot analysis, an empirical CLT demonstration, formal hypothesis testing, and linear,
logistic, spline, and Poisson regression — to two linked finance datasets. Each technique was applied
because the data genuinely supported it (e.g., spline regression only after decile-binned means confirmed
real nonlinearity; Poisson regression only after confirming a genuine count variable existed), not to
satisfy a checklist. The credit-risk model shows real but moderate predictive power, and the DTI policy
backtest illustrates that any resulting recommendation requires an explicit business trade-off, not a
p-value alone. All results are reported honestly, including unexpected or inconvenient findings (slower
CLT convergence, overdispersion, tiny effect sizes despite significance), consistent with the standard
a genuine consulting engagement would require.